In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/04.gold-helpers"

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructors_df = spark.table(f"{catalog_name}.{silver_schema}.constructors").filter((F.col("batch_id") == v_batch_id))
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_constructors_df = (
    constructors_df
    .join(ref_nationality_region_df, on='nationality', how='left')
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
    )
)

In [0]:
display(dim_constructors_df.head(5))

In [0]:
columns_to_update = [col for col in dim_constructors_df.columns if col not in ["constructor_id"]]
columns_to_update

In [0]:
write_to_gold(
    df=dim_constructors_df,
    target_table=target_table,
    columns_to_update=columns_to_update,
    merge_condition="t.constructor_id=s.constructor_id"
)

In [0]:
display(spark.table(target_table).limit(5))